# SPRING: Deriving the $N_p$-form from the $N_s$-form

**Reference:** Goldshlager, Abrahamsen, Lin (2024). *A Kaczmarz-inspired approach to accelerate the optimization of neural network wavefunctions.* arXiv:2401.10190. The $N_s$-form is Eq. 33 in the paper.

**Why two forms?** SPRING is one algorithm with two equivalent implementations:

- **$N_s$-form** (direct): solves an $N_s\times N_s$ Gram system via Cholesky. Natural for `minSR_solver_gpu` in `vmc_torch/GPU/vmc_modules.py:537`.
- **$N_p$-form** (iterative): solves the full $N_p\times N_p$ operator via MINRES. Natural for `distributed_minres_solver_gpu` in `vmc_torch/GPU/vmc_modules.py:280`.

Both produce identical $\phi_k$ in exact arithmetic. The $N_p$-form makes SPRING a **one-line** extension of our existing iterative MINRES solver: just add $\mu\lambda\,\phi_{k-1}$ to the right-hand side.

This notebook derives the equivalence and verifies it numerically.

## Notation

Let $N_s$ = minibatch size, $N_p$ = number of variational parameters. All quantities are real.

| Symbol | Shape | Meaning |
|---|---|---|
| $\bar O$ | $(N_s, N_p)$ | Centered, scaled log-amplitude gradients. Row $i$ is $\bigl[\nabla_\theta \log\psi_\theta(R_i) - \langle\nabla\log\psi\rangle\bigr]/\sqrt{N_s}$. |
| $e$ | $(N_s,)$ | Centered, scaled local energies: $(E_L(R_i) - \langle E_L\rangle)/\sqrt{N_s}$. |
| $g$ | $(N_p,)$ | Covariance gradient $g = \bar O^T e$. |
| $S$ | $(N_p, N_p)$ | $N_p$-space Gram (QGT): $S = \bar O^T \bar O$. |
| $T$ | $(N_s, N_s)$ | $N_s$-space Gram: $T = \bar O\bar O^T$. |
| $\lambda$ | scalar | Tikhonov damping. |
| $\mu$ | scalar | Kaczmarz decay (paper default $\approx 0.99$). |
| $\phi_{k-1}$ | $(N_p,)$ | SPRING iterate from the previous step. $\phi_0 = 0$. |

**Sign convention:** the paper uses $\bar\varepsilon = -\delta\tau(E_L-\langle E\rangle)/\sqrt{N_s}$; we fold the $-\delta\tau$ into the outer parameter update and work with $e = +(E_L-\langle E\rangle)/\sqrt{N_s}$. The flipped sign is cosmetic — it propagates to $\phi$ becoming the *gradient direction* (so our outer update is $\theta\leftarrow\theta - \eta\,\phi_k$), and doesn't affect any of the algebra below.

## Starting point: the $N_s$-form

In our sign convention, SPRING's Kaczmarz recurrence (paper Eq. 33) is

$$
\phi_k \;=\; \bar O^T\,(T + \lambda I)^{-1}\,(e - \mu\,\bar O\,\phi_{k-1}) \;+\; \mu\,\phi_{k-1}.
$$

Distribute the $\bar O^T(T+\lambda I)^{-1}$ across the parenthesis so the three pieces are visible separately:

$$
\phi_k \;=\; \underbrace{\bar O^T (T+\lambda I)^{-1}\,e}_{(\mathrm{i})}
        \;-\; \mu\,\underbrace{\bar O^T (T+\lambda I)^{-1}\,\bar O}_{(\mathrm{ii})}\,\phi_{k-1}
        \;+\; \mu\,\phi_{k-1}. \qquad (\ast)
$$

**Goal:** rewrite $(\ast)$ so that the only linear system is on the full $N_p$-dim space — i.e. expressible as $(S+\lambda I)^{-1}(\text{some vector})$, which an iterative MINRES solver can handle via matvecs without ever forming a dense matrix.

## Step 1 — Push-through identity

**Claim.** For any real $\bar O\in\mathbb{R}^{N_s\times N_p}$ and $\lambda > 0$,

$$
\bar O^T\,(\bar O\bar O^T + \lambda I_{N_s})^{-1} \;=\; (\bar O^T\bar O + \lambda I_{N_p})^{-1}\,\bar O^T.
$$

**Proof.** Start from the trivial equality

$$
\bar O^T\bar O\,\bar O^T \,+\, \lambda\,\bar O^T \;=\; \bar O^T\bar O\,\bar O^T \,+\, \lambda\,\bar O^T.
$$

Factor the LHS as $\bar O^T(\bar O\bar O^T + \lambda I_{N_s})$ and the RHS as $(\bar O^T\bar O + \lambda I_{N_p})\bar O^T$:

$$
\bar O^T\,(\bar O\bar O^T + \lambda I_{N_s}) \;=\; (\bar O^T\bar O + \lambda I_{N_p})\,\bar O^T.
$$

Left-multiply by $(\bar O^T\bar O + \lambda I_{N_p})^{-1}$ and right-multiply by $(\bar O\bar O^T + \lambda I_{N_s})^{-1}$:

$$
(\bar O^T\bar O + \lambda I_{N_p})^{-1}\,\bar O^T \;=\; \bar O^T\,(\bar O\bar O^T + \lambda I_{N_s})^{-1}. \qquad\blacksquare
$$

In short: $\bar O^T$ "pushes through" the inverse, moving from the $N_s$-space resolvent to the $N_p$-space resolvent. In our shorthand with $T=\bar O\bar O^T$ and $S=\bar O^T\bar O$:

$$
\bar O^T\,(T + \lambda I)^{-1} \;=\; (S + \lambda I)^{-1}\,\bar O^T.
$$

## Step 2 — Apply push-through to the two pieces

Apply the identity to $(\mathrm{i})$ and $(\mathrm{ii})$ from equation $(\ast)$:

**(i)** $\;\bar O^T(T+\lambda I)^{-1}\,e \;=\; (S+\lambda I)^{-1}\bar O^T e \;=\; (S+\lambda I)^{-1}\,g\;$ where $g \equiv \bar O^T e$.

**(ii)** $\;\bar O^T(T+\lambda I)^{-1}\,\bar O \;=\; (S+\lambda I)^{-1}\bar O^T \bar O \;=\; (S+\lambda I)^{-1}\,S.$

Both pieces now live entirely in $N_p$-space.

## Step 3 — Resolvent trick on $(S+\lambda I)^{-1} S$

Because $(S+\lambda I) - \lambda I = S$,

$$
(S+\lambda I)^{-1}\,S \;=\; (S+\lambda I)^{-1}\bigl[(S+\lambda I) - \lambda I\bigr] \;=\; I \;-\; \lambda\,(S+\lambda I)^{-1}.
$$

This is a single-line algebraic rearrangement — no SVD or eigendecomposition needed.

## Step 4 — Substitute back and simplify

Plug Steps 2 and 3 into $(\ast)$:

$$
\phi_k \;=\; (S+\lambda I)^{-1}\,g \;-\; \mu\bigl[I - \lambda(S+\lambda I)^{-1}\bigr]\phi_{k-1} \;+\; \mu\,\phi_{k-1}.
$$

Expand the middle bracket:

$$
\phi_k \;=\; (S+\lambda I)^{-1}\,g \;-\; \mu\,\phi_{k-1} \;+\; \mu\lambda\,(S+\lambda I)^{-1}\,\phi_{k-1} \;+\; \mu\,\phi_{k-1}.
$$

The $-\mu\phi_{k-1}$ and $+\mu\phi_{k-1}$ cancel:

$$
\phi_k \;=\; (S+\lambda I)^{-1}\,g \;+\; \mu\lambda\,(S+\lambda I)^{-1}\,\phi_{k-1}.
$$

Factor the shared $(S+\lambda I)^{-1}$:

$$
\boxed{\;\phi_k \;=\; (S + \lambda I)^{-1}\,\bigl(g + \mu\lambda\,\phi_{k-1}\bigr)\;}
$$

This is the **$N_p$-form**.

## Interpretation

Compare to plain MinSR in our sign convention:

$$
\phi_k^{\mathrm{MinSR}} \;=\; (S + \lambda I)^{-1}\,g.
$$

**SPRING differs from MinSR by exactly one thing** — the RHS picks up a $\mu\lambda\,\phi_{k-1}$ term. Same operator $(S+\lambda I)$, same matvec, same iterative solver. Only the right-hand side changes.

Two sanity checks:

1. **$\mu = 0$** $\Longrightarrow$ SPRING collapses exactly to MinSR. This is a bit-level correctness test — set `spring_mu=0` and confirm output matches the baseline MINRES solver.
2. **$\lambda \to 0$** $\Longrightarrow$ the $\mu\lambda\,\phi_{k-1}$ term vanishes and SPRING also collapses to MinSR. This mirrors the paper's observation that SPRING's "momentum" is *regularization-induced* — it's fundamentally tied to Tikhonov damping, unlike naive momentum (MinSR+M, paper Eq. 43) where $\mu$ acts independently of $\lambda$.

## Implementation in our GPU solvers

### Iterative variant — `spring_minres_solver_gpu` (new)

One-line change vs. `distributed_minres_solver_gpu` (`vmc_torch/GPU/vmc_modules.py:280`). The existing `gpu_matvec` already applies the centered operator $(S + \lambda I)$. Reuse it unchanged and only modify the RHS:

```python
# Before (MINRES, solves  S*dp = energy_grad):
dp, info = torch_minres(gpu_matvec, energy_grad, rtol=rtol, maxiter=maxiter)

# After (SPRING, solves  S*phi_k = energy_grad + mu*lam*phi_prev):
rhs = energy_grad + (mu * diag_shift) * phi_prev_gpu
dp, info = torch_minres(gpu_matvec, rhs, rtol=rtol, maxiter=maxiter)
phi_prev_gpu = dp   # persist across iterations
```

### Direct variant — `spring_minsr_solver_gpu` (new)

Two-op change vs. `minSR_solver_gpu` (`vmc_torch/GPU/vmc_modules.py:537`). The direct Cholesky solver works natively in the $N_s$-form:

```python
# Before (minSR):
alpha = torch.linalg.solve(T, E_s)
dp = lpg_scaled.T @ alpha

# After (SPRING, Ns-form):
rhs = E_s - mu * (lpg_scaled @ phi_prev_gpu)
alpha = torch.linalg.solve(T, rhs)
dp = lpg_scaled.T @ alpha + mu * phi_prev_gpu
phi_prev_gpu = dp
```

Both produce identical $\phi_k$ in exact arithmetic — verified numerically below.

## Numerical verification

Build a small random $\bar O$, $e$, $\phi_{k-1}$, then compute $\phi_k$ via both forms and compare.

In [1]:
import torch

torch.manual_seed(0)

# Toy dimensions (Ns < Np — the typical MinSR/SPRING regime)
N_s, N_p = 16, 200
lam = 1e-3          # Tikhonov damping
mu  = 0.9           # Kaczmarz decay

# Random centered O-bar, e, and previous phi.
# Center O-bar's columns (removes the 1-nullspace) and scale by 1/sqrt(Ns)
# to match the paper's convention.
O_bar = torch.randn(N_s, N_p, dtype=torch.float64)
O_bar -= O_bar.mean(dim=0, keepdim=True)
O_bar /= (N_s ** 0.5)

e = torch.randn(N_s, dtype=torch.float64)
e -= e.mean()
e /= (N_s ** 0.5)

phi_prev = torch.randn(N_p, dtype=torch.float64)

# ---- N_s form (direct Cholesky / solve) --------------------------------
T = O_bar @ O_bar.T + lam * torch.eye(N_s, dtype=torch.float64)
rhs_s = e - mu * (O_bar @ phi_prev)
alpha = torch.linalg.solve(T, rhs_s)
phi_Ns = O_bar.T @ alpha + mu * phi_prev

# ---- N_p form ----------------------------------------------------------
# (direct solve for the check; in production this is iterative MINRES)
S = O_bar.T @ O_bar + lam * torch.eye(N_p, dtype=torch.float64)
g = O_bar.T @ e
rhs_p = g + mu * lam * phi_prev
phi_Np = torch.linalg.solve(S, rhs_p)

# ---- Compare -----------------------------------------------------------
diff = torch.linalg.vector_norm(phi_Ns - phi_Np).item()
rel_diff = diff / torch.linalg.vector_norm(phi_Ns).item()
print(f'||phi_Ns - phi_Np||   = {diff:.3e}')
print(f'relative difference   = {rel_diff:.3e}')
print(f'match within 1e-10?    {"YES" if rel_diff < 1e-10 else "NO"}')

assert rel_diff < 1e-10, 'Np-form and Ns-form disagree — derivation bug!'

||phi_Ns - phi_Np||   = 9.129e-12
relative difference   = 7.972e-13
match within 1e-10?    YES


### Sanity check 2: $\mu = 0$ collapses to MinSR

Setting $\mu = 0$ should reduce both SPRING forms to plain MinSR (in our sign convention), independent of $\phi_{k-1}$:

$$
\phi_k \;=\; (S+\lambda I)^{-1}\,g \;=\; \bar O^T\,(T+\lambda I)^{-1}\,e.
$$

In [2]:
# mu = 0 should reduce both SPRING forms to plain MinSR,
# independent of whatever phi_prev we pass in.
phi_prev_any = torch.randn(N_p, dtype=torch.float64)
mu0 = 0.0

# N_s form with mu=0
rhs_s_minsr = e - mu0 * (O_bar @ phi_prev_any)
phi_minsr_Ns = O_bar.T @ torch.linalg.solve(T, rhs_s_minsr) + mu0 * phi_prev_any

# N_p form with mu=0
rhs_p_minsr = g + mu0 * lam * phi_prev_any
phi_minsr_Np = torch.linalg.solve(S, rhs_p_minsr)

# Reference: plain MinSR
phi_minsr_ref = O_bar.T @ torch.linalg.solve(T, e)

err_Ns = torch.linalg.vector_norm(phi_minsr_Ns - phi_minsr_ref).item()
err_Np = torch.linalg.vector_norm(phi_minsr_Np - phi_minsr_ref).item()
print(f'||Ns-form@mu=0  -  MinSR ref|| = {err_Ns:.3e}')
print(f'||Np-form@mu=0  -  MinSR ref|| = {err_Np:.3e}')
assert err_Ns < 1e-10 and err_Np < 1e-10

||Ns-form@mu=0  -  MinSR ref|| = 0.000e+00
||Np-form@mu=0  -  MinSR ref|| = 6.950e-13


## Appendix: why the $\omega P$ regularization is free in the $N_p$-form

The paper adds an extra $\omega P$ term to the $N_s$-space matrix (Sec. 3.2, Eq. 39):

$$
\phi_k^{\mathrm{paper}} \;=\; \bar O^T\,(T + \lambda I + \omega P)^{-1}\,\bar\zeta_k \;+\; \mu\,\phi_{k-1}, \qquad P = \tfrac{1}{N_s}\mathbf{1}\mathbf{1}^T.
$$

This is a pure numerical stabilizer: $T$ is always singular because $\bar O$ is centered ($T\mathbf{1}=0$), so a floating-point perturbation to $T$ can push its smallest eigenvalue below $-\lambda$, making $T+\lambda I$ indefinite and breaking Cholesky.

**Claim.** The $\omega P$ term does not change $\phi_k$.

**Proof.** $\mathbf{1}$ is an eigenvector of $(T+\lambda I)$ with eigenvalue $\lambda$ (because $T\mathbf{1}=0$), and also an eigenvector of $(T+\lambda I+\omega P)$ with eigenvalue $\omega+\lambda$. By Woodbury (paper Eq. 40),

$$
(T+\lambda I+\omega P)^{-1} \;=\; (T+\lambda I)^{-1} \;+\; \left(\tfrac{1}{\omega+\lambda} - \tfrac{1}{\lambda}\right)\,P.
$$

Left-multiplying by $\bar O^T$ annihilates the extra $P$ term, because the columns of $\bar O$ are centered so $\sum_i \bar O_{ij} = 0$ for every $j$, i.e. $\mathbf{1}^T\bar O = 0$, hence $\bar O^T P = \tfrac{1}{N_s}\bar O^T\mathbf{1}\mathbf{1}^T = 0$:

$$
\bar O^T\,(T+\lambda I+\omega P)^{-1} \;=\; \bar O^T\,(T+\lambda I)^{-1}.\quad\blacksquare
$$

**Consequence:** the $\omega P$ stabilizer is a *Cholesky-only concern*. It's invisible to the $N_p$-form solver (MINRES on $(S+\lambda I)$), so `spring_minres_solver_gpu` needs no extra term. For `spring_minsr_solver_gpu` (direct Cholesky path) it can be added as $+\omega P$ on the $\mathbf{1}$-direction if numerical instability shows up in practice, but it can be deferred until we actually observe a failure.